In [0]:
# 01b — Bronze: Demanda (cortes de boleto por empresa) + puente GTFS

import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"

API_BASE = "https://gobiernoabierto.cordoba.gob.ar/api/datos-abiertos/dato"
GTFS_DATASET_ID = 3319
CORTES_BOLETO_DATASET_ID = 3318
CORTES_BOLETO_VERSION_ID = 6118  # "por empresa (datos diarios) - 2018/2023"

# ID del Google Sheet resuelto en la exploración — confirmar que sigue vigente
CORTES_BOLETO_SHEET_ID = "1kOD1hpb1-5Ul6A9e9aujxhCg8BEBm8SL9D3wV2YeaUg"
# GID de la pestaña "Boletos por día" — copiar del URL del navegador (?gid=...)
# al parar sobre esa pestaña específica del Sheet
CORTES_BOLETO_SHEET_GID = "0"  # TODO: reemplazar por el gid real de "Boletos por día"

In [0]:
## 1. Resolver la versión y los recursos de "Cortes de boleto" (3318)
resp = requests.get(f"{API_BASE}/{CORTES_BOLETO_DATASET_ID}/version-dato", timeout=30)
resp.raise_for_status()
versiones = resp.json()["results"]
 
print(f"Versiones publicadas: {len(versiones)}")
for v in versiones:
    print(f"  - id={v['id']} | titulo={v.get('titulo')} | fecha={v.get('fecha_creacion', v.get('fecha_publicacion', '?'))}")
 
version_actual = versiones[0]
print(f"\nUsando versión más reciente: {version_actual}")


In [0]:
## 1. Cortes de boleto diarios por empresa (Google Sheet)
sheet_url_completa = recursos_v2[0]["url"]
sheet_id = sheet_url_completa.split("/d/")[1].split("/")[0]
sheet_gid = sheet_url_completa.split("gid=")[1]

print(f"sheet_id: {sheet_id}")
print(f"sheet_gid: {sheet_gid}")

sheet_csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}"

resp = requests.get(sheet_csv_url, timeout=60)
resp.raise_for_status()
print(f"Content-Type: {resp.headers.get('content-type')}")

cortes_pdf = pd.read_csv(io.BytesIO(resp.content))
print(f"Columnas: {list(cortes_pdf.columns)}")
cortes_pdf.head()

In [0]:
audit = {
    "_ingested_at": datetime.now(timezone.utc),
    "_source_dataset_id": str(CORTES_BOLETO_DATASET_ID),
    "_source_url": sheet_csv_url,  # no es una URL firmada, se puede guardar completa
}
for col, val in audit.items():
    cortes_pdf[col] = val

spark.createDataFrame(cortes_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.cortes_boleto_diario")

count = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.cortes_boleto_diario").count()
print(f"cortes_boleto_diario: {count} filas escritas.")

In [0]:
## 2. Puente GTFS: agency.txt, trips.txt, stop_times.txt

resp = requests.get(f"{API_BASE}/{GTFS_DATASET_ID}/version-dato", timeout=30)
resp.raise_for_status()
gtfs_version = resp.json()["results"][0]

resp = requests.get(f"{API_BASE}/{GTFS_DATASET_ID}/version-dato/{gtfs_version['id']}/recurso", timeout=30)
resp.raise_for_status()
gtfs_url = resp.json()["results"][0]["url"]

gtfs_zip = zipfile.ZipFile(io.BytesIO(requests.get(gtfs_url, timeout=60).content))
print("Archivos en el GTFS:", gtfs_zip.namelist())

audit_gtfs = {
    "_ingested_at": datetime.now(timezone.utc),
    "_source_dataset_id": str(GTFS_DATASET_ID),
    "_source_url": gtfs_url.split("?")[0],
}

In [0]:
trips_pdf = pd.read_csv(gtfs_zip.open("trips.txt"))
for col, val in audit_gtfs.items():
    trips_pdf[col] = val
spark.createDataFrame(trips_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_trips")
print(f"gtfs_trips: {len(trips_pdf)} filas escritas.")

In [0]:
agency_pdf = pd.read_csv(gtfs_zip.open("agency.txt"))
for col, val in audit_gtfs.items():
    agency_pdf[col] = val
spark.createDataFrame(agency_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_agency")
print(f"gtfs_agency: {len(agency_pdf)} filas escritas.")
display(agency_pdf)

In [0]:
stop_times_pdf = pd.read_csv(gtfs_zip.open("stop_times.txt"))
for col, val in audit_gtfs.items():
    stop_times_pdf[col] = val
spark.createDataFrame(stop_times_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_stop_times")
print(f"gtfs_stop_times: {len(stop_times_pdf)} filas escritas.")

In [0]:
for tabla in ["cortes_boleto_diario", "gtfs_agency", "gtfs_trips", "gtfs_stop_times"]:
    count = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabla}").count()
    print(f"{CATALOG}.{SCHEMA_BRONZE}.{tabla}: {count} filas")